# SciFact GTE Reranker Fine-Tuning

This notebook fine-tunes `Alibaba-NLP/gte-reranker-modernbert-base` for the SciFact claim-to-abstract task.

Training target:
- `SUPPORT` -> relevant
- `CONTRADICT` -> relevant
- `NOINFO` -> not relevant

That matches the retrieval objective you care about: a good reranker should surface abstracts that either support the claim or refute it. Generic web/search rerankers often underweight contradiction evidence in scientific abstracts, which is one likely reason the `embeddinggemma_fact_check` retriever baseline stayed so strong.


In [ ]:
# Optional: clone the repo fresh in Colab.
# !git clone <YOUR_REPO_URL> /content/ScholarRAG


In [1]:
%cd /content/ScholarRAG
!python -m pip install --upgrade pip
!python -m pip install -e . scikit-learn


/content/ScholarRAG
Obtaining file:///content/ScholarRAG
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for scholarrag (pyproject.toml) ... done
  Created wheel for scholarrag: filename=scholarrag-0.1.0-0.editable-py3-none-any.whl size=3296 sha256=c74684cab03ac03160162c0e4a1788d41197484999fdcc241165502112008987
  Stored in directory: /tmp/pip-ephem-wheel-cache-c4ybq1o_/wheels/0a/89/0c/d5413d54f0d9857abad3277dd48a8e9f3f1a399dd033320dd6
Successfully built scholarrag
  Attempting uninstall: scholarrag
    Found existing installation: scholarrag 0.1.0
    Uninstalling scholarrag-0.1.0:
      Successfully uninstalled scholarrag-0.1.0


In [2]:
!nvidia-smi


Thu Apr 30 20:39:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   41C    P8             17W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Label setup

SciFact source claims include cited documents and evidence annotations. For each `(claim, cited abstract)` pair:
- if the abstract has a `SUPPORT` evidence annotation, we label it `1`
- if the abstract has a `CONTRADICT` evidence annotation, we also label it `1`
- if the abstract is cited but has no evidence annotation for that claim, we label it `0` as `NOINFO`

That means this fine-tune explicitly teaches the reranker that both accepting and refuting evidence are relevant retrieval targets.


In [14]:
from datetime import datetime
from pathlib import Path
import json
import os

BASE_MODEL_ID = "Alibaba-NLP/gte-reranker-modernbert-base"
RUN_NAME = f"scifact_gte_reranker_ft_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
OUTPUT_DIR = Path("experiments/scifact/results") / RUN_NAME
MODEL_OUTPUT_DIR = OUTPUT_DIR / "model"
HF_EXPORT_DIR = OUTPUT_DIR / "hf_export"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
HF_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Keep Trainer logging local in Colab and avoid interactive W&B prompts.
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"

TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
MAX_LENGTH = 1024
EPOCHS = 10
LEARNING_RATE = 2e-5
WARMUP_RATIO = 0.1
USE_AMP = True
SEED = 42

CONFIG = {
    "base_model_id": BASE_MODEL_ID,
    "output_dir": str(OUTPUT_DIR),
    "model_output_dir": str(MODEL_OUTPUT_DIR),
    "hf_export_dir": str(HF_EXPORT_DIR),
    "train_batch_size": TRAIN_BATCH_SIZE,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "max_length": MAX_LENGTH,
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "warmup_ratio": WARMUP_RATIO,
    "use_amp": USE_AMP,
    "seed": SEED,
}
print(json.dumps(CONFIG, indent=2))


{
  "base_model_id": "Alibaba-NLP/gte-reranker-modernbert-base",
  "output_dir": "experiments/scifact/results/scifact_gte_reranker_ft_20260430_213843",
  "model_output_dir": "experiments/scifact/results/scifact_gte_reranker_ft_20260430_213843/model",
  "hf_export_dir": "experiments/scifact/results/scifact_gte_reranker_ft_20260430_213843/hf_export",
  "train_batch_size": 16,
  "eval_batch_size": 32,
  "max_length": 1024,
  "epochs": 10,
  "learning_rate": 2e-05,
  "warmup_ratio": 0.1,
  "use_amp": true,
  "seed": 42
}


In [2]:
from collections import Counter

import pandas as pd
from datasets import load_dataset

PARQUET_BASE = "hf://datasets/allenai/scifact_entailment@refs/convert/parquet/default/"
TRAIN_PATH = PARQUET_BASE + "train/*.parquet"
VALIDATION_PATH = PARQUET_BASE + "validation/*.parquet"

claims_train = load_dataset("parquet", data_files=TRAIN_PATH, split="train")
claims_val = load_dataset("parquet", data_files=VALIDATION_PATH, split="train")

def join_abstract(abstract):
    if abstract is None:
        return ""
    if isinstance(abstract, list):
        return " ".join(sentence.strip() for sentence in abstract if sentence and sentence.strip())
    return str(abstract).strip()

def build_document_text(title, abstract):
    title = (title or "").strip()
    abstract_text = join_abstract(abstract)
    if title and abstract_text:
        return f"{title}\n\n{abstract_text}"
    return title or abstract_text

def verdict_to_binary(verdict):
    verdict = str(verdict).upper()
    if verdict in {"SUPPORT", "CONTRADICT", "REFUTE", "REFUTES"}:
        return 1.0
    return 0.0

def build_pair_frame(claim_rows):
    pair_rows = []
    verdict_counts = Counter()
    for row in claim_rows:
        verdict = str(row["verdict"]).upper()
        verdict_counts[verdict] += 1
        pair_rows.append({
            "claim_id": int(row["claim_id"]),
            "doc_id": int(row["abstract_id"]),
            "claim": row["claim"].strip(),
            "title": (row.get("title") or "").strip(),
            "document_text": build_document_text(row.get("title"), row.get("abstract")),
            "verdict": verdict,
            "binary_label": verdict_to_binary(verdict),
        })
    frame = pd.DataFrame(pair_rows)
    return frame, verdict_counts

train_frame, train_verdict_counts = build_pair_frame(claims_train)
val_frame, val_verdict_counts = build_pair_frame(claims_val)

summary = pd.DataFrame([
    {
        "split": "train",
        "rows": len(train_frame),
        "positive_rows": int(train_frame["binary_label"].sum()),
        "negative_rows": int((1.0 - train_frame["binary_label"]).sum()),
        "support_rows": int(train_verdict_counts.get("SUPPORT", 0)),
        "contradict_rows": int(train_verdict_counts.get("CONTRADICT", 0)),
        "noinfo_rows": int(train_verdict_counts.get("NOINFO", 0)),
    },
    {
        "split": "validation",
        "rows": len(val_frame),
        "positive_rows": int(val_frame["binary_label"].sum()),
        "negative_rows": int((1.0 - val_frame["binary_label"]).sum()),
        "support_rows": int(val_verdict_counts.get("SUPPORT", 0)),
        "contradict_rows": int(val_verdict_counts.get("CONTRADICT", 0)),
        "noinfo_rows": int(val_verdict_counts.get("NOINFO", 0)),
    },
])
display(summary)
display(train_frame.head(30))


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


,split,rows,positive_rows,negative_rows,support_rows,contradict_rows,noinfo_rows
0,train,919,564,355,370,194,0
1,validation,340,209,131,138,71,0


,claim_id,doc_id,claim,title,document_text,verdict,binary_label
0,0,31715818,0-dimensional biomaterials lack inductive prop...,New opportunities: the use of nanotechnologies...,New opportunities: the use of nanotechnologies...,NEI,0.0
1,2,13734012,1 in 5 million in UK have abnormal PrP positiv...,Prevalent abnormal prion protein in human appe...,Prevalent abnormal prion protein in human appe...,CONTRADICT,1.0
2,4,22942787,1-1% of colorectal cancer patients are diagnos...,Relation between Medicare screening reimbursem...,Relation between Medicare screening reimbursem...,NEI,0.0
3,6,2613775,10% of sudden infant death syndrome (SIDS) dea...,Sudden infant death syndrome.,Sudden infant death syndrome.\n\nDespite decli...,NEI,0.0
4,9,44265107,32% of liver transplantation programs required...,Liver transplantation and opioid dependence.,Liver transplantation and opioid dependence.\n...,SUPPORT,1.0
5,10,32587939,4-PBA treatment decreases endoplasmic reticulu...,Wolfram syndrome 1 and adenylyl cyclase 8 inte...,Wolfram syndrome 1 and adenylyl cyclase 8 inte...,NEI,0.0
6,11,32587939,4-PBA treatment raises endoplasmic reticulum s...,Wolfram syndrome 1 and adenylyl cyclase 8 inte...,Wolfram syndrome 1 and adenylyl cyclase 8 inte...,NEI,0.0
7,12,33409100,40mg/day dosage of folic acid and 2mg/day dosa...,Effect of homocysteine lowering on mortality a...,Effect of homocysteine lowering on mortality a...,SUPPORT,1.0
8,14,641786,5'-nucleotidase metabolizes 6MP.,Relapse specific mutations in NT5C2 in childho...,Relapse specific mutations in NT5C2 in childho...,NEI,0.0
9,15,22080671,50% of patients exposed to radiation have acti...,KLF4-dependent phenotypic modulation of smooth...,KLF4-dependent phenotypic modulation of smooth...,NEI,0.0


In [3]:
from sentence_transformers import InputExample
from torch.utils.data import DataLoader

train_examples = [
    InputExample(texts=[row.claim, row.document_text], label=float(row.binary_label))
    for row in train_frame.itertuples(index=False)
]
val_pairs = list(zip(val_frame["claim"].tolist(), val_frame["document_text"].tolist(), strict=True))
val_labels = val_frame["binary_label"].astype(int).tolist()

train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=TRAIN_BATCH_SIZE)
print({
    "train_examples": len(train_examples),
    "val_examples": len(val_pairs),
    "train_batches": len(train_dataloader),
})


{'train_examples': 919, 'val_examples': 340, 'train_batches': 58}


In [15]:
import math
import random
import time

import numpy as np
import torch
from sentence_transformers import CrossEncoder

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

warmup_steps = math.ceil(len(train_dataloader) * EPOCHS * WARMUP_RATIO)
model = CrossEncoder(
    BASE_MODEL_ID,
    num_labels=1,
    max_length=MAX_LENGTH,
    trust_remote_code=True,
)

start_time = time.perf_counter()
model.fit(
    train_dataloader=train_dataloader,
    epochs=EPOCHS,
    warmup_steps=warmup_steps,
    optimizer_params={"lr": LEARNING_RATE},
    output_path=str(MODEL_OUTPUT_DIR),
    use_amp=USE_AMP,
    show_progress_bar=True,
)
model.model.save_pretrained(HF_EXPORT_DIR)
model.tokenizer.save_pretrained(HF_EXPORT_DIR)
training_seconds = time.perf_counter() - start_time
print({
    "model_output_dir": str(MODEL_OUTPUT_DIR),
    "hf_export_dir": str(HF_EXPORT_DIR),
    "training_seconds": round(training_seconds, 2),
    "warmup_steps": warmup_steps,
})


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': None, 'bos_token_id': None}.


Step,Training Loss
500,0.078500


{'model_output_dir': 'experiments/scifact/results/scifact_gte_reranker_ft_20260430_213843/model', 'hf_export_dir': 'experiments/scifact/results/scifact_gte_reranker_ft_20260430_213843/hf_export', 'training_seconds': 316.9, 'warmup_steps': 58}


In [25]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, average_precision_score, precision_recall_fscore_support, roc_auc_score
from sentence_transformers import CrossEncoder

def evaluate_cross_encoder(model_name, cross_encoder, pairs, labels):
    scores = np.asarray(
        cross_encoder.predict(pairs, batch_size=EVAL_BATCH_SIZE, show_progress_bar=True),
        dtype=float,
    ).reshape(-1)
    probs = 1.0 / (1.0 + np.exp(-scores))
    preds = (probs >= 0.5).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary", zero_division=0)
    metrics = {
        "model": model_name,
        "roc_auc": float(roc_auc_score(labels, probs)),
        "average_precision": float(average_precision_score(labels, probs)),
        "accuracy": float(accuracy_score(labels, preds)),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
    }
    return metrics, probs, preds

print("Loading base model for validation comparison...")
base_model = CrossEncoder(
    BASE_MODEL_ID,
    num_labels=1,
    max_length=MAX_LENGTH,
    trust_remote_code=True,
)
base_metrics, base_probs, base_preds = evaluate_cross_encoder("base_gte", base_model, val_pairs, val_labels)

if not HF_EXPORT_DIR.exists():
    raise FileNotFoundError(
        f"Fine-tuned Hugging Face export not found at {HF_EXPORT_DIR}. Run the training cell first so the export is written."
    )

print(f"Loading fine-tuned model from {HF_EXPORT_DIR}...")
trained_model = CrossEncoder(
    str(HF_EXPORT_DIR),
    num_labels=1,
    max_length=MAX_LENGTH,
    trust_remote_code=True,
)
ft_metrics, ft_probs, ft_preds = evaluate_cross_encoder("finetuned_gte", trained_model, val_pairs, val_labels)

comparison = pd.DataFrame([base_metrics, ft_metrics])
comparison["delta_vs_base"] = [0.0] + [ft_metrics["average_precision"] - base_metrics["average_precision"]]

val_report = val_frame.copy()
val_report["base_score"] = base_probs
val_report["base_pred_label"] = base_preds
val_report["finetuned_score"] = ft_probs
val_report["finetuned_pred_label"] = ft_preds

verdict_score_summary = val_report.groupby("verdict").agg(
    count=("verdict", "size"),
    base_score_mean=("base_score", "mean"),
    finetuned_score_mean=("finetuned_score", "mean"),
    base_score_median=("base_score", "median"),
    finetuned_score_median=("finetuned_score", "median"),
).reset_index()

with (OUTPUT_DIR / "validation_comparison.json").open("w") as handle:
    json.dump(comparison.to_dict(orient="records"), handle, indent=2)
val_report.to_csv(OUTPUT_DIR / "validation_predictions.csv", index=False)
display(comparison)
display(verdict_score_summary)


Loading base model for validation comparison...


Batches:   0%|          | 0/11 [00:00<?, ?it/s]

Loading fine-tuned model from experiments/scifact/results/scifact_gte_reranker_ft_20260430_213843/hf_export...


Batches:   0%|          | 0/11 [00:00<?, ?it/s]

,model,roc_auc,average_precision,accuracy,precision,recall,f1,delta_vs_base
0,base_gte,0.874466,0.912605,0.614706,0.614706,1.0,0.761384,0.000000
1,finetuned_gte,0.941561,0.954950,0.614706,0.614706,1.0,0.761384,0.042345


,verdict,count,base_score_mean,finetuned_score_mean,base_score_median,finetuned_score_median
0,CONTRADICT,71,0.705089,0.689600,0.708235,0.731038
1,NEI,131,0.685907,0.529341,0.688555,0.500004
2,SUPPORT,138,0.714641,0.721238,0.717019,0.731057


In [26]:
print("Saved model:", MODEL_OUTPUT_DIR)
print("HF export:", HF_EXPORT_DIR)
print("Validation comparison:", OUTPUT_DIR / "validation_comparison.json")
print("Validation predictions:", OUTPUT_DIR / "validation_predictions.csv")
# Optional: zip the run directory for download from Colab.
# !zip -r "{OUTPUT_DIR}.zip" "{OUTPUT_DIR}"


Saved model: experiments/scifact/results/scifact_gte_reranker_ft_20260430_213843/model
HF export: experiments/scifact/results/scifact_gte_reranker_ft_20260430_213843/hf_export
Validation comparison: experiments/scifact/results/scifact_gte_reranker_ft_20260430_213843/validation_comparison.json
Validation predictions: experiments/scifact/results/scifact_gte_reranker_ft_20260430_213843/validation_predictions.csv
